# 04_resume.ipynb
Résumé extractif (sumy TextRank) et option abstractive (mT5).
Export d'une fonction réutilisable summary_tool.py dans ged-ai/.


In [ ]:
from time import time
import pandas as pd
from utils import ARTIFACTS_DIR, load_documents_csv

# sumy extractive (TextRank)
try:
    from sumy.parsers.plaintext import PlaintextParser
    from sumy.nlp.tokenizers import Tokenizer
    from sumy.summarizers.text_rank import TextRankSummarizer
except Exception:
    raise SystemExit('Installer sumy: pip install sumy')

def summarise_extractive(text: str, sentences_count: int = 3) -> str:
    if not text or not text.strip():
        return ''
    parser = PlaintextParser.from_string(text, Tokenizer('french'))
    summarizer = TextRankSummarizer()
    summary = summarizer(parser.document, sentences_count)
    return '\n'.join(str(s) for s in summary)

# Optional abstractive via transformers (mT5)
abstractive_available = True
try:
    from transformers import pipeline
    abstractive = pipeline('summarization', model='csebuetnlp/mT5_multilingual_XLSum', device=-1)
except Exception as e:
    abstractive_available = False
    print('Abstractive model not available locally (pip install transformers and download model). Will skip abstractive tests.')

def summarise_abstractive(text: str, max_length: int = 150) -> str:
    if not abstractive_available:
        raise RuntimeError('Abstractive summarizer not available')
    # wrapper to call pipeline (may be slow)
    out = abstractive(text, max_length=max_length, min_length=40, do_sample=False)
    return out[0]['summary_text']

# Compare speed/quality on 10 documents
df = load_documents_csv('documents_extraits.csv')
sample = df['texte_nettoye'].dropna().astype(str).tolist()[:10]
results = []
for i, doc in enumerate(sample):
    r = {'i': i}
    t0 = time(); s_ext = summarise_extractive(doc, sentences_count=3); t1 = time();
    r['extractive_time'] = t1-t0; r['extractive'] = s_ext
    if abstractive_available:
        t0 = time(); s_abs = summarise_abstractive(doc, max_length=150); t1 = time();
        r['abstractive_time'] = t1-t0; r['abstractive'] = s_abs
    results.append(r)

res_df = pd.DataFrame(results)
out_path = ARTIFACTS_DIR + '/summaries_comparison.csv'
res_df.to_csv(out_path, index=False)
print('Saved comparison to', out_path)

# Export fonction réutilisable vers summary_tool.py
summary_tool_path = os.path.join(os.path.dirname(__file__), 'summary_tool.py')
code = '''from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer

def summarise_extractive(text: str, sentences_count: int = 3) -> str:
    if not text or not text.strip():
        return ''
    parser = PlaintextParser.from_string(text, Tokenizer('french'))
    summarizer = TextRankSummarizer()
    summary = summarizer(parser.document, sentences_count)
    return '\n'.join(str(s) for s in summary)
'''
# write file
with open(summary_tool_path, 'w', encoding='utf8') as f:
    f.write(code)
print('Wrote', summary_tool_path)
